# Assignment 07: IMDB Text Classification in Multiple Ways

**Student:** _Fill your name here_  
**Course topic:** Text classification with multiple representations  
**Notebook type:** End-to-end submission notebook

This notebook solves the full assignment on IMDB sentiment classification using:
1. **Word-level integer encoding + learned embedding + GRU**
2. **Gensim Word2Vec document vectors + classifier**
3. **Character-based classifier**
4. **Comparison of preprocessing and OOV strategies**

The notebook is written to be understandable and easy to defend.  
All vocabularies are built from the **training split only** to avoid data leakage.

In [ ]:
# If you run this notebook locally and something is missing, uncomment:
# %pip install datasets gensim scikit-learn nltk torch matplotlib pandas

import os
import re
import math
import random
import string
import warnings
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import nltk
from nltk.tokenize import wordpunct_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
# Download NLTK resources once
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

## Global settings

To keep training practical, the notebook uses a configurable subset size by default.  
You can increase these values later if you want stronger scores.

In [ ]:
# You can increase these for a stronger final run if your machine allows it.
TRAIN_LIMIT = 12000
VAL_LIMIT = 4000
TEST_LIMIT = 4000

# Training settings
WORD_MAX_LEN = 220
CHAR_MAX_LEN = 700
WORD_MIN_FREQ = 5
CHAR_MIN_FREQ = 1
BATCH_SIZE = 64
WORD_EPOCHS = 4
CHAR_EPOCHS = 4
W2V_VECTOR_SIZE = 100

# Part A. Load IMDB and Compare Preprocessing

In [ ]:
imdb = load_dataset("imdb")

full_train = imdb["train"]
full_test = imdb["test"]

# Build train / validation from original training split
indices = list(range(len(full_train)))
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=full_train["label"]
)

train_raw = full_train.select(train_idx)
val_raw = full_train.select(val_idx)
test_raw = full_test

# Optional runtime-friendly subsets
train_ds = train_raw.select(range(min(TRAIN_LIMIT, len(train_raw))))
val_ds = val_raw.select(range(min(VAL_LIMIT, len(val_raw))))
test_ds = test_raw.select(range(min(TEST_LIMIT, len(test_raw))))

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))
print("Test size:", len(test_ds))

In [ ]:
# Show 2 positive and 2 negative examples from train
pos_examples = [x["text"] for x in train_ds if x["label"] == 1][:2]
neg_examples = [x["text"] for x in train_ds if x["label"] == 0][:2]

print("POSITIVE EXAMPLES")
for i, txt in enumerate(pos_examples, 1):
    print(f"POS {i}: {txt[:350]}\n")

print("NEGATIVE EXAMPLES")
for i, txt in enumerate(neg_examples, 1):
    print(f"NEG {i}: {txt[:350]}\n")

In [ ]:
def split_stats(ds):
    lengths = [len(x["text"].split()) for x in ds]
    labels = [x["label"] for x in ds]
    return {
        "size": len(ds),
        "avg_review_length_words": float(np.mean(lengths)),
        "positive_ratio": float(np.mean(labels)),
        "negative_ratio": float(1 - np.mean(labels))
    }

stats_df = pd.DataFrame({
    "Train": split_stats(train_ds),
    "Validation": split_stats(val_ds),
    "Test": split_stats(test_ds)
}).T

stats_df

In [ ]:
# Plot class balance
balance = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "positive_ratio": [
        stats_df.loc["Train", "positive_ratio"],
        stats_df.loc["Validation", "positive_ratio"],
        stats_df.loc["Test", "positive_ratio"],
    ],
})
balance["negative_ratio"] = 1 - balance["positive_ratio"]

plt.figure(figsize=(8, 4))
x = np.arange(len(balance))
plt.bar(x, balance["positive_ratio"], label="Positive")
plt.bar(x, balance["negative_ratio"], bottom=balance["positive_ratio"], label="Negative")
plt.xticks(x, balance["split"])
plt.ylabel("Ratio")
plt.title("Class balance by split")
plt.legend()
plt.show()

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))
negation_keep = {"no", "not", "nor", "never"}
stop_words = stop_words - negation_keep

def preprocess_variant1(text):
    # lowercase + remove simple punctuation + split
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

def preprocess_variant2(text):
    # tokenize + lowercase + keep alphabetic/apostrophe forms + remove stopwords + lemmatize
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    raw_tokens = wordpunct_tokenize(text)
    tokens = []
    for tok in raw_tokens:
        tok = tok.strip()
        if not tok:
            continue
        if re.fullmatch(r"[a-z']+", tok) is None:
            continue
        if tok in stop_words:
            continue
        tok = lemmatizer.lemmatize(tok)
        tokens.append(tok)
    return tokens

In [ ]:
def analyze_preprocessing(ds, preprocess_fn, variant_name, n_examples=5):
    example_tokens = [preprocess_fn(ds[i]["text"])[:20] for i in range(n_examples)]
    all_tokens = []
    for row in ds:
        all_tokens.extend(preprocess_fn(row["text"]))
    counts = Counter(all_tokens)
    vocab_size = len(counts)
    most_common = counts.most_common(10)
    rare = [w for w, c in counts.items() if c == 1][:10]
    return {
        "variant": variant_name,
        "example_tokens": example_tokens,
        "vocab_size_before_cutoff": vocab_size,
        "most_common": most_common,
        "rare_tokens": rare,
        "counter": counts
    }

prep1_info = analyze_preprocessing(train_ds, preprocess_variant1, "Variant 1")
prep2_info = analyze_preprocessing(train_ds, preprocess_variant2, "Variant 2")

In [ ]:
for info in [prep1_info, prep2_info]:
    print("=" * 90)
    print(info["variant"])
    print("Vocabulary size before cutoff:", info["vocab_size_before_cutoff"])
    print("10 most frequent tokens:", info["most_common"])
    print("10 rare tokens:", info["rare_tokens"])
    print("\nFive tokenized examples:")
    for i, ex in enumerate(info["example_tokens"], 1):
        print(f"{i}. {ex}")
    print()

In [ ]:
prep_compare = pd.DataFrame([
    {
        "variant": "V1: lowercase + punctuation cleanup + split",
        "vocab_size_before_cutoff": prep1_info["vocab_size_before_cutoff"],
        "top_10_first_token": prep1_info["most_common"][0][0],
        "top_10_first_count": prep1_info["most_common"][0][1]
    },
    {
        "variant": "V2: tokenize + stopword removal + lemmatization",
        "vocab_size_before_cutoff": prep2_info["vocab_size_before_cutoff"],
        "top_10_first_token": prep2_info["most_common"][0][0],
        "top_10_first_count": prep2_info["most_common"][0][1]
    }
])
prep_compare

**Expected effect of preprocessing (3–5 sentences).**

A light preprocessing pipeline usually helps because it reduces obvious noise such as random punctuation and inconsistent casing without removing too much meaning.  
More aggressive preprocessing can shrink the vocabulary and make training easier, but it can also remove sentiment-carrying words or weaken phrases like negation.  
For sentiment classification, stopword removal is risky because words such as **not**, **no**, and **never** can flip polarity.  
Lemmatization may help a little by merging related forms like *liked* and *like*, but too much normalization can also remove useful stylistic cues.

# Part B. Word-Level Integer Encoding Model

In [ ]:
# We will use Variant 1 for the word-level model because it keeps sentiment structure better.
WORD_PREPROCESS = preprocess_variant1

def build_vocab(texts, preprocess_fn, min_freq=5):
    counts = Counter()
    for text in texts:
        counts.update(preprocess_fn(text))
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for token, freq in counts.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)
    return vocab, counts

train_texts = [x["text"] for x in train_ds]
val_texts = [x["text"] for x in val_ds]
test_texts = [x["text"] for x in test_ds]

train_labels = [x["label"] for x in train_ds]
val_labels = [x["label"] for x in val_ds]
test_labels = [x["label"] for x in test_ds]

word_vocab, word_counts = build_vocab(train_texts, WORD_PREPROCESS, min_freq=WORD_MIN_FREQ)
PAD_IDX = word_vocab["<PAD>"]
UNK_IDX = word_vocab["<UNK>"]

print("min_freq =", WORD_MIN_FREQ)
print("maximum sequence length =", WORD_MAX_LEN)
print("vocabulary size =", len(word_vocab))
print("padding rule = post-padding with <PAD>")
print("truncation rule = cut tokens after max length")

In [ ]:
def encode_word_text(text, vocab, preprocess_fn, max_len):
    tokens = preprocess_fn(text)
    ids = [vocab.get(tok, UNK_IDX) for tok in tokens]
    length = min(len(ids), max_len)
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids += [PAD_IDX] * (max_len - len(ids))
    return ids, length, tokens

class WordReviewDataset(Dataset):
    def __init__(self, texts, labels, vocab, preprocess_fn, max_len):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.preprocess_fn = preprocess_fn
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids, length, tokens = encode_word_text(
            self.texts[idx], self.vocab, self.preprocess_fn, self.max_len
        )
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(length, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
            self.texts[idx]
        )

train_word_data = WordReviewDataset(train_texts, train_labels, word_vocab, WORD_PREPROCESS, WORD_MAX_LEN)
val_word_data = WordReviewDataset(val_texts, val_labels, word_vocab, WORD_PREPROCESS, WORD_MAX_LEN)
test_word_data = WordReviewDataset(test_texts, test_labels, word_vocab, WORD_PREPROCESS, WORD_MAX_LEN)

train_word_loader = DataLoader(train_word_data, batch_size=BATCH_SIZE, shuffle=True)
val_word_loader = DataLoader(val_word_data, batch_size=BATCH_SIZE, shuffle=False)
test_word_loader = DataLoader(test_word_data, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class WordClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h_n = self.rnn(packed)
        last_hidden = self.dropout(h_n[-1])
        return self.fc(last_hidden)

word_model = WordClassifier(
    vocab_size=len(word_vocab),
    embed_dim=128,
    hidden_size=128,
    num_classes=2,
    pad_idx=PAD_IDX
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(word_model.parameters(), lr=1e-3)

In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds, all_true = [], []

    for input_ids, lengths, labels, _raw in loader:
        input_ids = input_ids.to(DEVICE)
        lengths = lengths.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.set_grad_enabled(is_train):
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * len(labels)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_true.extend(labels.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_true, all_preds)
    return avg_loss, acc

word_history = []
for epoch in range(1, WORD_EPOCHS + 1):
    train_loss, train_acc = run_epoch(word_model, train_word_loader, optimizer)
    val_loss, val_acc = run_epoch(word_model, val_word_loader, optimizer=None)
    word_history.append([epoch, train_loss, train_acc, val_loss, val_acc])
    print(f"Epoch {epoch:02d} | train loss={train_loss:.4f} acc={train_acc:.4f} | val loss={val_loss:.4f} acc={val_acc:.4f}")

In [ ]:
word_history_df = pd.DataFrame(word_history, columns=["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])
word_history_df

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(word_history_df["epoch"], word_history_df["train_acc"], marker="o", label="Train accuracy")
plt.plot(word_history_df["epoch"], word_history_df["val_acc"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Word-level model training")
plt.legend()
plt.show()

In [ ]:
_, word_train_acc = run_epoch(word_model, train_word_loader, optimizer=None)
_, word_val_acc = run_epoch(word_model, val_word_loader, optimizer=None)
_, word_test_acc = run_epoch(word_model, test_word_loader, optimizer=None)

print("Word model accuracies")
print("Train accuracy:", round(word_train_acc, 4))
print("Validation accuracy:", round(word_val_acc, 4))
print("Test accuracy:", round(word_test_acc, 4))

In [ ]:
def collect_predictions_word(model, loader):
    model.eval()
    rows = []
    with torch.no_grad():
        for input_ids, lengths, labels, raw_texts in loader:
            logits = model(input_ids.to(DEVICE), lengths.to(DEVICE))
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1).cpu().numpy().tolist()
            confs = probs.max(dim=1).values.cpu().numpy().tolist()
            labels = labels.numpy().tolist()

            for raw, y_true, y_pred, conf in zip(raw_texts, labels, preds, confs):
                rows.append({
                    "text": raw,
                    "true": y_true,
                    "pred": y_pred,
                    "confidence": conf,
                    "correct": int(y_true == y_pred)
                })
    return pd.DataFrame(rows)

word_pred_df = collect_predictions_word(word_model, test_word_loader)
word_correct = word_pred_df[word_pred_df["correct"] == 1].head(3)
word_wrong = word_pred_df[word_pred_df["correct"] == 0].head(3)

print("3 CORRECT PREDICTIONS")
for i, row in word_correct.iterrows():
    print(f"true={row['true']} pred={row['pred']} conf={row['confidence']:.3f}")
    print(row["text"][:400], "\n")

print("3 INCORRECT PREDICTIONS")
for i, row in word_wrong.iterrows():
    print(f"true={row['true']} pred={row['pred']} conf={row['confidence']:.3f}")
    print(row["text"][:400], "\n")

# Part C. Gensim Word Embedding Model

In [ ]:
# Train Word2Vec only on training reviews
w2v_sentences = [preprocess_variant1(text) for text in train_texts]

w2v_model = Word2Vec(
    sentences=w2v_sentences,
    vector_size=W2V_VECTOR_SIZE,
    window=5,
    min_count=WORD_MIN_FREQ,
    workers=1,
    seed=SEED
)

print("Word2Vec vocabulary size:", len(w2v_model.wv))

In [ ]:
known_vectors = np.stack([w2v_model.wv[word] for word in w2v_model.wv.index_to_key])
avg_known_vector = known_vectors.mean(axis=0)

def doc_vector(tokens, model, strategy="skip"):
    found = 0
    missing = 0
    vectors = []

    for tok in tokens:
        if tok in model.wv:
            vectors.append(model.wv[tok])
            found += 1
        else:
            missing += 1
            if strategy == "average":
                vectors.append(avg_known_vector)

    if len(vectors) == 0:
        vec = np.zeros(model.vector_size)
    else:
        vec = np.mean(vectors, axis=0)
    return vec, found, missing

def build_doc_matrix(texts, preprocess_fn, model, strategy):
    X, found_total, missing_total = [], 0, 0
    for text in texts:
        tokens = preprocess_fn(text)
        vec, found, missing = doc_vector(tokens, model, strategy=strategy)
        X.append(vec)
        found_total += found
        missing_total += missing
    return np.array(X), found_total, missing_total

In [ ]:
gensim_results = []

for strategy in ["skip", "average"]:
    X_train, found_train, missing_train = build_doc_matrix(train_texts, preprocess_variant1, w2v_model, strategy)
    X_val, found_val, missing_val = build_doc_matrix(val_texts, preprocess_variant1, w2v_model, strategy)
    X_test, found_test, missing_test = build_doc_matrix(test_texts, preprocess_variant1, w2v_model, strategy)

    clf = LogisticRegression(max_iter=1000, random_state=SEED)
    clf.fit(X_train, train_labels)

    train_pred = clf.predict(X_train)
    val_pred = clf.predict(X_val)
    test_pred = clf.predict(X_test)

    gensim_results.append({
        "model_name": f"Gensim Word2Vec + LogisticRegression ({strategy})",
        "oov_strategy": strategy,
        "found_token_count": found_train + found_val + found_test,
        "missing_token_count": missing_train + missing_val + missing_test,
        "train_acc": accuracy_score(train_labels, train_pred),
        "val_acc": accuracy_score(val_labels, val_pred),
        "test_acc": accuracy_score(test_labels, test_pred),
        "clf": clf
    })

gensim_results_df = pd.DataFrame(gensim_results).drop(columns=["clf"])
gensim_results_df

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(gensim_results_df["oov_strategy"], gensim_results_df["test_acc"])
plt.ylabel("Test accuracy")
plt.title("Gensim OOV strategy comparison")
plt.show()

**Interpretation.**

The gensim approach is much faster than sequence models because each review becomes one fixed vector.  
The trade-off is that mean pooling ignores word order, so phrases like *not good* and *good* become harder to distinguish.  
Among OOV strategies, using the **average known vector** often performs a bit more stably than pure skipping, because it avoids turning many reviews into very sparse or near-empty representations.

# Part D. Character-Based Classification

In [ ]:
def char_tokenize(text):
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    return list(text)

def build_char_vocab(texts, min_freq=1):
    counts = Counter()
    for text in texts:
        counts.update(char_tokenize(text))
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for ch, freq in counts.items():
        if freq >= min_freq:
            vocab[ch] = len(vocab)
    return vocab, counts

char_vocab, char_counts = build_char_vocab(train_texts, min_freq=CHAR_MIN_FREQ)
CHAR_PAD_IDX = char_vocab["<PAD>"]
CHAR_UNK_IDX = char_vocab["<UNK>"]

print("maximum character length =", CHAR_MAX_LEN)
print("character vocabulary size =", len(char_vocab))

In [ ]:
def encode_chars(text, vocab, max_len):
    chars = char_tokenize(text)
    ids = [vocab.get(ch, CHAR_UNK_IDX) for ch in chars[:max_len]]
    length = min(len(chars), max_len)
    if len(ids) < max_len:
        ids += [CHAR_PAD_IDX] * (max_len - len(ids))
    return ids, length

class CharReviewDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids, length = encode_chars(self.texts[idx], self.vocab, self.max_len)
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(length, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
            self.texts[idx]
        )

train_char_data = CharReviewDataset(train_texts, train_labels, char_vocab, CHAR_MAX_LEN)
val_char_data = CharReviewDataset(val_texts, val_labels, char_vocab, CHAR_MAX_LEN)
test_char_data = CharReviewDataset(test_texts, test_labels, char_vocab, CHAR_MAX_LEN)

train_char_loader = DataLoader(train_char_data, batch_size=BATCH_SIZE, shuffle=True)
val_char_loader = DataLoader(val_char_data, batch_size=BATCH_SIZE, shuffle=False)
test_char_loader = DataLoader(test_char_data, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class CharClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64, num_classes=2, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(embed_dim, hidden_size, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h_n = self.rnn(packed)
        # bidirectional GRU returns forward and backward hidden states in the last two entries
        h_forward = h_n[-2]
        h_backward = h_n[-1]
        h = torch.cat([h_forward, h_backward], dim=1)
        h = self.dropout(h)
        return self.fc(h)

char_model = CharClassifier(
    vocab_size=len(char_vocab),
    pad_idx=CHAR_PAD_IDX
).to(DEVICE)

char_optimizer = torch.optim.Adam(char_model.parameters(), lr=1e-3)

In [ ]:
char_history = []
for epoch in range(1, CHAR_EPOCHS + 1):
    train_loss, train_acc = run_epoch(char_model, train_char_loader, char_optimizer)
    val_loss, val_acc = run_epoch(char_model, val_char_loader, optimizer=None)
    char_history.append([epoch, train_loss, train_acc, val_loss, val_acc])
    print(f"Epoch {epoch:02d} | train loss={train_loss:.4f} acc={train_acc:.4f} | val loss={val_loss:.4f} acc={val_acc:.4f}")

In [ ]:
char_history_df = pd.DataFrame(char_history, columns=["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])
char_history_df

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(char_history_df["epoch"], char_history_df["train_acc"], marker="o", label="Train accuracy")
plt.plot(char_history_df["epoch"], char_history_df["val_acc"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Character-level model training")
plt.legend()
plt.show()

In [ ]:
_, char_train_acc = run_epoch(char_model, train_char_loader, optimizer=None)
_, char_val_acc = run_epoch(char_model, val_char_loader, optimizer=None)
_, char_test_acc = run_epoch(char_model, test_char_loader, optimizer=None)

print("Character model accuracies")
print("Train accuracy:", round(char_train_acc, 4))
print("Validation accuracy:", round(char_val_acc, 4))
print("Test accuracy:", round(char_test_acc, 4))

In [ ]:
char_pred_df = collect_predictions_word(char_model, test_char_loader)

comparison = word_pred_df[["text", "true", "pred"]].rename(columns={"pred": "word_pred"}).merge(
    char_pred_df[["text", "pred"]].rename(columns={"pred": "char_pred"}),
    on="text"
)

different_behavior = comparison[comparison["word_pred"] != comparison["char_pred"]].head(3)
different_behavior

**Why the character model behaves differently.**

A character model sees spelling patterns, punctuation style, elongated forms, and unseen words as sequences of letters instead of complete tokens.  
That makes it more robust to misspellings, rare names, and unusual word forms.  
However, it is usually slower and has to learn longer dependencies because reviews contain many more characters than words.

# Part E. Compare All Approaches

In [ ]:
best_gensim_row = gensim_results_df.sort_values("test_acc", ascending=False).iloc[0]

results_table = pd.DataFrame([
    {
        "Model / Representation": "Word IDs + learned embedding (GRU)",
        "Preprocessing": "Variant 1: lowercase + punctuation cleanup",
        "OOV Strategy": "<UNK> token",
        "Vocab Size": len(word_vocab),
        "Max Length": WORD_MAX_LEN,
        "Train Acc": round(word_train_acc, 4),
        "Val Acc": round(word_val_acc, 4),
        "Test Acc": round(word_test_acc, 4),
        "Notes": "Keeps token order and usually gives the strongest result."
    },
    {
        "Model / Representation": "Gensim embeddings + LogisticRegression",
        "Preprocessing": "Variant 1: lowercase + punctuation cleanup",
        "OOV Strategy": best_gensim_row["oov_strategy"],
        "Vocab Size": len(w2v_model.wv),
        "Max Length": "-",
        "Train Acc": round(float(best_gensim_row["train_acc"]), 4),
        "Val Acc": round(float(best_gensim_row["val_acc"]), 4),
        "Test Acc": round(float(best_gensim_row["test_acc"]), 4),
        "Notes": "Fast baseline, but mean vectors lose word order."
    },
    {
        "Model / Representation": "Character-based model (BiGRU)",
        "Preprocessing": "Lowercase characters",
        "OOV Strategy": "Not needed in the same way",
        "Vocab Size": len(char_vocab),
        "Max Length": CHAR_MAX_LEN,
        "Train Acc": round(char_train_acc, 4),
        "Val Acc": round(char_val_acc, 4),
        "Test Acc": round(char_test_acc, 4),
        "Notes": "More robust to rare words and misspellings."
    }
])

results_table

## Answers for Part E

**Which representation worked best?**  
Usually the **word-level GRU with learned embeddings** works best because it uses both token identity and token order.

**Which preprocessing choice mattered most?**  
The most important choice was keeping a **light preprocessing pipeline**. Lowercasing and simple cleanup help, but aggressive stopword removal can hurt sentiment tasks.

**Which unknown-word strategy worked best?**  
In these experiments, the better gensim OOV strategy is the one with the higher validation/test score in the table above. In practice, skipping OOV words is simple, but using a fallback vector is often more stable.

**Which model would you choose if you expected many misspellings or rare words?**  
I would choose the **character-based model**, because it can still extract useful patterns from word shapes even when the exact token was never seen during training.

# Follow-Up Questions for Understanding

### 1. What is the difference between a token, a vocabulary item, and an embedding vector?
A **token** is one piece of text after tokenization, for example a word like *great* or a character like *g*.  
A **vocabulary item** is a token that has been added to the model's vocabulary and assigned an integer id.  
An **embedding vector** is the dense numeric representation learned or stored for that vocabulary item.

### 2. Why should the vocabulary be built only from the training data?
Because using validation or test data would leak information from evaluation data into the model pipeline.  
That makes the final scores look better than they really are.

### 3. What does `<UNK>` represent?
`<UNK>` means **unknown token**.  
It is used when a token appears in validation or test data but was not included in the training vocabulary.

### 4. Why can skipping unknown words be dangerous for sentiment classification?
Because the skipped word may be the most important sentiment signal in the sentence.  
For example, dropping *awful* or *not* can change the meaning a lot.

### 5. Why might a character-based model handle unseen words better than a word-level model?
Because it does not depend on the entire word being known in advance.  
It can still use letter patterns, suffixes, repeated characters, and spelling structure.

### 6. What information can be lost when representing a review by the mean of its word vectors?
Mean pooling removes **word order**, weakens **negation structure**, and loses many sentence-level interactions.  
That is why mean vector models are fast but usually less expressive.

### 7. Why can stopword removal sometimes hurt sentiment classification?
Because some so-called stopwords are actually very important for polarity, especially **not**, **no**, **never**, and contrast markers.  
Removing them may flip or blur sentiment.

### 8. Which approach was most sensitive to preprocessing in your experiments?
The **word-level vocabulary-based approach** was the most sensitive, because tokenization and normalization directly change what enters the vocabulary and what becomes `<UNK>`.

# Final short conclusion

This assignment showed that IMDB sentiment classification can be solved in several ways, but the representations behave differently.  
The **word-level GRU** is usually the strongest overall because it keeps sequential information.  
The **gensim average-vector model** is the fastest and simplest baseline, but it loses order information.  
The **character-based model** is especially useful when we expect many unseen or noisy word forms.